### 02.- Ingest Transactions
Ingesta incremental del archivo de transacciones (CSV) desde la capa RAW hacia Bronze.



In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container",       "raw")
dbutils.widgets.text("containerDestino","bronze")
dbutils.widgets.text("catalogo",        "bank_dev")
dbutils.widgets.text("esquema",         "bronze")
dbutils.widgets.text("storageName",     "saprojectbankdeveastus")

In [0]:
container        = dbutils.widgets.get("container")
containerDestino = dbutils.widgets.get("containerDestino")
catalogo         = dbutils.widgets.get("catalogo")
esquema          = dbutils.widgets.get("esquema")
storageName      = dbutils.widgets.get("storageName")

In [0]:
storage_path         = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
storage_path_destino = f"abfss://{containerDestino}@{storageName}.dfs.core.windows.net/"

raw_transactions_root            = f"{storage_path}external/transactions/"
schema_location_transactions     = f"{storage_path_destino}_schemas/transactions"
checkpoint_location_transactions = f"{storage_path_destino}_checkpoints/transactions"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

## Transactions

In [0]:
raw_schema_transactions = StructType([
    StructField("id",             StringType()),
    StructField("date",           StringType()),
    StructField("client_id",      StringType()),
    StructField("card_id",        StringType()),
    StructField("amount",         StringType()),  # puede venir con "$" o comas
    StructField("use_chip",       StringType()),
    StructField("merchant_id",    StringType()),
    StructField("merchant_city",  StringType()),
    StructField("merchant_state", StringType()),
    StructField("zip",            StringType()),  # nunca numeric: pierde ceros iniciales
    StructField("mcc",            StringType()),
    StructField("errors",         StringType()),
])

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location_transactions)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .schema(raw_schema_transactions)
    .load(raw_transactions_root)
)

df_bronze = df_stream.withColumn("ingestion_date", F.current_timestamp())

query_transactions = (
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location_transactions)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{catalogo}.{esquema}.transactions")
)